# Milestone 2 - Fine-tune Stable Diffusion with LoRA

This notebook builds on the pretrained text-conditional Stable Diffusion 1.5 pipeline. It freezes the CLIP text encoder, VAE, and pretrained U-Net weights, and trains small LoRA adapters on the U-Net attention projections using only Flickr30k training pairs. The baseline remains in notebook 03 for a later same-prompt comparison.

Architecture: caption -> CLIP tokenizer -> 77 token hidden states -> U-Net cross-attention with a noisy image latent -> denoised latent -> VAE decoder -> image. See `docs/m2_integration_architecture.md` for the full data flow and training details.

## 1. Environment and prerequisites

Use a CUDA GPU runtime (a Colab T4 is recommended). Clone/open this repository and run notebook cells from its root directory. Run notebook 01 first so `data/flickr30k_subset/metadata.csv` and `data/flickr30k_subset/images/` are available. Notebook 03 is the unchanged pretrained baseline; run it before fine-tuning when collecting comparison results.

In [ ]:
!pip install -q "diffusers==0.41.0" "transformers==5.20.1" accelerate peft pandas pillow safetensors

In [ ]:
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Select a CUDA GPU runtime before training."
assert Path("scripts/train_sd_lora.py").is_file(), "Run this notebook from the repository root."
assert Path("data/flickr30k_subset/metadata.csv").is_file(), "Run notebook 01 to prepare the dataset first."
print("GPU:", torch.cuda.get_device_name(0))
print("Dataset and training script found.")

## 2. Train U-Net LoRA adapters

The default is a bounded first experiment: 800 optimizer updates, batch size 1, rank 8, and learning rate 1e-4. Raise `TRAIN_STEPS` only after reviewing the first run's loss and held-out generations. The script checks the dataset and GPU explicitly and writes the adapter, configuration, and loss log to an ignored `models/` directory.

In [ ]:
TRAIN_STEPS = 800
OUTPUT_DIR = Path("models/sd15-flickr30k-lora")

!python scripts/train_sd_lora.py --train-steps {TRAIN_STEPS} --output-dir models/sd15-flickr30k-lora

## 3. Load the adapter and preview a validation caption

This sanity check loads the saved adapter into the original pretrained pipeline and generates one validation caption. It does not replace the systematic baseline-versus-fine-tuned evaluation; use identical prompts and seeds for that comparison.

In [ ]:
import pandas as pd
from diffusers import StableDiffusionPipeline
from IPython.display import display

BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
dtype = torch.float16
pipeline = StableDiffusionPipeline.from_pretrained(BASE_MODEL, torch_dtype=dtype).to("cuda")
pipeline.load_lora_weights(str(OUTPUT_DIR))
pipeline.set_progress_bar_config(disable=True)

metadata = pd.read_csv("data/flickr30k_subset/metadata.csv")
validation = metadata.loc[metadata["split"] == "val"].reset_index(drop=True)
assert not validation.empty, "Dataset has no validation captions."
preview_prompt = validation.loc[0, "caption"]
generator = torch.Generator(device="cuda").manual_seed(42)
preview = pipeline(preview_prompt, num_inference_steps=30, guidance_scale=7.5, generator=generator).images[0]
preview.save(OUTPUT_DIR / "validation_preview.png")
print(preview_prompt)
display(preview)